In [ ]:
import numpy as np
from skimage.util import view_as_windows
import os
import shutil
import gc
from tensorflow.data import Dataset
from tensorflow import TensorSpec
import h5py
from sklearn.utils import shuffle
from tensorflow.keras import layers,models
from tensorflow.data import AUTOTUNE
from tensorflow import optimizers
from tensorflow import losses
from tensorflow.keras import callbacks
from tensorflow.keras import backend
from tensorflow.keras import regularizers


In [ ]:
def extract_patches(image, patch_size=(7, 7), stride=5, depth=224):
    h, w, _ = image.shape
    m, n = patch_size
    if m > h or n > w:
        raise ValueError("Patch size is larger than image dimensions")

    pad_h = m // 2
    pad_w = n // 2
    padded_image = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode='reflect')

    patches = view_as_windows(padded_image, (m, n, depth), step=stride)
    num_patches = patches.shape[0] * patches.shape[1]
    patches = patches.reshape(num_patches, m, n, depth)

    patches = np.transpose(patches, (0, 3, 1, 2))
    patches = patches[..., np.newaxis]

    return patches

In [ ]:
def extract_patches(image, patch_size=(7, 7), stride=5):
    h, w, u = image.shape
    m, n = patch_size
    if m > h or n > w:
        raise ValueError("Patch size is larger than image dimensions")
    pad_h = m // 2
    pad_w = n // 2
    padded_image = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode='reflect')
    patches = view_as_windows(padded_image, (m, n, u), step=stride)
    patches = patches.reshape(-1, 1, m, n, u)
    return patches


In [ ]:
imgs = [
    "/content/drive/MyDrive/hsi data/canola/canola_1.npy",
    "/content/drive/MyDrive/hsi data/canola/canola_2.npy",
    "/content/drive/MyDrive/hsi data/redroot/redroot_pigweed_1.npy",
    "/content/drive/MyDrive/hsi data/redroot/redroot_pigweed_2.npy",
    "/content/drive/MyDrive/hsi data/kochia/tmp/kochia_1.npy",
    "/content/drive/MyDrive/hsi data/kochia/tmp/kochia_2.npy",
    "/content/drive/MyDrive/hsi data/waterhemp/tmp/waterhemp_1.npy",
    "/content/drive/MyDrive/hsi data/waterhemp/tmp/waterhemp_2.npy",
    "/content/drive/MyDrive/hsi data/sugarbeet/sugarbeet_1.npy",
    "/content/drive/MyDrive/hsi data/sugarbeet/sugarbeet_2.npy",
    "/content/drive/MyDrive/hsi data/soybean/soybean_1.npy",
    "/content/drive/MyDrive/hsi data/soybean/soybean_2.npy",
    "/content/drive/MyDrive/hsi data/ragweed/ragweed_1.npy",
    "/content/drive/MyDrive/hsi data/ragweed/ragweed_2.npy",
    "/content/drive/MyDrive/hsi data/waterhemp/tmp/waterhemp_3.npy",
]

class_mapping = {
    "canola": 0,
    "kochia": 1,
    "ragweed": 2,
    "redroot": 3,
    "soybean": 4,
    "sugarbeet": 5,
    "waterhemp": 6,
}


hdf5_path = "/content/X_y_patches2.h5"
n_samples = len(imgs)

with h5py.File(hdf5_path, "w") as f:
    for i, path in enumerate(imgs):
        img = np.load(path)
        patches = extract_patches(img)
        num_patches = patches.shape[0]
        patch_shape = patches.shape[1:]

        class_label = os.path.splitext(os.path.basename(path))[0].split("_")[0]
        class_id = class_mapping[class_label]
        print(patches.shape)
        f.create_dataset(f"X_{i}", data=patches, dtype="float32")
        f.create_dataset(f"y_{i}", data=np.array(class_id, dtype="int32"))



In [ ]:
train_images = [0, 2, 4, 6, 8, 10, 12] 
val_images = [1, 3, 5, 7, 9, 11]
hdf5_path = "/content/X_y_patches2.h5"

def make_generator(hdf5_path, image_indices):
    def generator():
        with h5py.File(hdf5_path, "r") as f:
            for idx in image_indices:
                patches = f[f"X_{idx}"][:]
                label = f[f"y_{idx}"][()]
                labels = np.full(patches.shape[0], label, dtype=np.int32)
                yield patches, labels
    return generator

dataset_train = Dataset.from_generator(
    make_generator(hdf5_path, train_images),
    output_signature=(
        TensorSpec(shape=(None, 224, 7, 7, 1), dtype=np.float32),
        TensorSpec(shape=(None,), dtype=np.int32)
    )
).flat_map(lambda x, y: Dataset.from_tensor_slices((x, y))).shuffle(buffer_size=100000).batch(32).prefetch(AUTOTUNE)
dataset_val = Dataset.from_generator(
    make_generator(hdf5_path, val_images),
    output_signature=(
        TensorSpec(shape=(None, 224, 7, 7, 1), dtype=np.float32),
        TensorSpec(shape=(None,), dtype=np.int32)
    )
).flat_map(lambda x, y: Dataset.from_tensor_slices((x, y))).shuffle(buffer_size=100000).batch(32).prefetch(AUTOTUNE)

In [ ]:
import tensorflow as tf
class PosCoding(layers.Layer):
  def __init__(self, N, D):
    super(PosCoding,self).__init__()
    self.N = N
    self.D = D

  def build(self, inputs_shape):
    encodings = []
    for i in range(self.N):
      enc = []
      for j in range(self.D):
        pos = float(i)
        pow_ = j//2
        if(j%2==0):
          enc.append(tf.math.sin(pos/10000**(2*pow_/self.D)))
        else:
          enc.append(tf.math.cos(pos/10000**(2*pow_/self.D)))
      encodings.append(enc)
    self.encodings = tf.stack(encodings, axis=0)
  def call(self, inputs):
    return self.encodings + inputs


In [ ]:
class Pad(layers.Layer):
    def call(self, x):
        pad_depth = 1
        return tf.pad(x, [[0, 0], [pad_depth, pad_depth], [0, 0], [0, 0], [0, 0]], mode='reflect')
class Transpose(layers.Layer):
    def call(self, x):
      return tf.transpose(x,perm = [0,2,3,1,4])
    def compute_output_shape(self,input_shape):
      return (input_shape[0],input_shape[2],input_shape[3],input_shape[1],input_shape[4])
def build_model(input_shape = (224,7,7,1)):
  input = layers.Input(shape = input_shape)
  padded_input = Pad()(input)
  conv3d_1 = layers.Conv3D(32,(3,3,3),strides=(1,1,1),activation = "relu",data_format="channels_last")(padded_input)
  conv3d_1 = Pad()(conv3d_1)
  conv3d_1 = layers.Conv3D(64,(3,3,3),strides=(1,1,1),activation = "relu")(conv3d_1)

  pool = layers.AveragePooling3D((1,2,2),strides =(1,2,2))(input)
  conv3d_2 = layers.Conv3D(32,(3,3,3),strides=(1,1,1), padding="same", activation = "relu")(pool)
  conv3d_2 = layers.Conv3D(64,(3,3,3),strides=(1,1,1), padding="same", activation = "relu")(conv3d_2)

  D = 129*224
  N= 9

  conv3d_2 = Transpose()(conv3d_2)
  conv3d_1 = Transpose()(conv3d_1)
  pool = Transpose()(pool)

  transf_inp = layers.Concatenate()([pool,conv3d_1,conv3d_2])
  transf_inp = layers.Reshape((N,D))(transf_inp)
  transf_inp = layers.Conv1D(D//200,3,padding = 'same', activation='relu')(transf_inp)
  D=D//200
  transf_inp = PosCoding(N,D)(transf_inp)

  norm = layers.LayerNormalization()(transf_inp)
  transf = layers.MultiHeadAttention(4,D//4)(query = norm, key = norm, value = norm)
  transf = layers.Add()([transf, transf_inp])
  ff_conv = layers.Conv1D(D//2,5,padding = 'same', activation='relu')(transf)
  ff_conv = layers.Conv1D(D,5,padding = 'same', activation='relu')(ff_conv)
  ff_conv = layers.Add()([ff_conv,transf])

  norm = layers.LayerNormalization()(ff_conv)
  transf = layers.MultiHeadAttention(4,D//4)(query = norm, key = norm, value = norm)
  transf = layers.Add()([transf, transf_inp])
  ff_conv = layers.Conv1D(D//2,5,padding = 'same', activation='relu')(transf)
  ff_conv = layers.Conv1D(D,5,padding = 'same', activation='relu')(ff_conv)
  ff_conv = layers.Add()([ff_conv,transf])

  flat = layers.Flatten()(ff_conv)
  flat = layers.Dense(200,activation='relu', kernel_regularizer=regularizers.l2(0.01))(flat)

  outputs = layers.Dense(7,'softmax')(flat)

  model = models.Model(input,outputs)
  return model

In [ ]:
model = build_model()
print(model.summary())


In [ ]:
opt = optimizers.Adam(learning_rate = 0.0001)
model.compile(optimizer=opt,loss = losses.SparseCategoricalCrossentropy(),metrics = ["accuracy"])

chechpoint = callbacks.ModelCheckpoint("/content/drive/MyDrive/hsi data/transf_ver1.weights.h5",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only = True)
tb = callbacks.TensorBoard(
    log_dir="/content/logs",
    histogram_freq=1)

In [ ]:
history = model.fit(
    dataset_train,
    validation_data=dataset_val,
    epochs=10,
    callbacks=[chechpoint, tb]
)

In [ ]:
model.load_weights("/content/drive/MyDrive/hsi data/transf_ver1.weights.h5")

In [ ]:
#from tensorflow.keras import load_model
model = models.load_model("/content/drive/MyDrive/hsi data/ver4.keras")

In [ ]:
def full_prediction(data_path,  image_indices):
  def make_test_generator(hdf5_path, image_indices):
    def generator():
        with h5py.File(hdf5_path, "r") as f:
            for idx in image_indices:
                patches = f[f"X_{idx}"][:]
                preds = model.predict(patches)
                yield preds
    return generator
  def create_aggregated_model():
    inputs = layers.Input(shape = (None,7))
    x = layers.GlobalAveragePooling1D()(inputs)
    outputs = layers.Activation("softmax")(x)
    model = models.Model(inputs,outputs)
    return model
  class_mapping = {
    "canola": 0, "kochia": 1, "ragweed": 2, "redroot": 3,
    "soybean": 4, "sugarbeet": 5, "waterhemp": 6
  }


  infmodel = create_aggregated_model()
  test_dataset = Dataset.from_generator(
      make_test_generator(data_path, image_indices),
      output_signature=TensorSpec(shape=(None, 7), dtype=np.float32)
  ).batch(1)
  final = infmodel.predict(test_dataset)
  predicted_classes = np.argmax(final, axis=1)
  predicted_labels = [list(class_mapping.keys())[c] for c in predicted_classes]
  print("Predicted classes:", predicted_labels)
  return predicted_labels

In [ ]:
p = full_prediction("/content/X_y_patches2.h5",[0,1,2,3])

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'],markersize=20)
plt.plot(history.history['val_accuracy'], markersize=20)
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()
# plotting of training and validation loss curves
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()